# AI Agents Observability & Monitoring

## Why Observability Matters

Agents are **non-deterministic** and **multi-step** without visibility you can't:
- Debug why an agent gave a wrong answer
- Know which tool calls are failing
- Track costs across thousands of runs
- Detect prompt regressions after updates
- Measure latency at each step

---

## Observability Stack

```
Agent Run
│
├── Traces: Full execution tree (LLM calls, tool calls, timing)
├── Spans:  Individual operations within a trace
├── Logs:   Structured events (errors, decisions)
└── Metrics: Aggregated stats (latency P50/P95, cost, error rate)
```

---

## LangSmith

LangChain's observability platform:
- **Tracing**: Automatic trace capture for LangChain/LangGraph
- **Datasets**: Store input/output pairs for evaluation
- **Evaluations**: Run automated evaluators on traces
- **Monitoring**: Production dashboards
- **Prompt versioning**: Track prompt changes over time

Set up: Just set `LANGCHAIN_TRACING_V2=true` and `LANGCHAIN_API_KEY`.

---

## Cost Tracking

Track token usage and cost per run:

| Model | Input (per 1M) | Output (per 1M) |
|-------|---------------|----------------|
| GPT-4o | $2.50 | $10.00 |
| GPT-4o-mini | $0.15 | $0.60 |
| Claude Sonnet 4.6 | $3.00 | $15.00 |
| Claude Haiku 4.5 | $0.25 | $1.25 |

$$\text{cost} = \frac{\text{input\_tokens}}{10^6} \times \text{price\_in} + \frac{\text{output\_tokens}}{10^6} \times \text{price\_out}$$

---

## Evaluation Metrics for Agents

| Metric | Description |
|--------|-------------|
| **Task Success Rate** | % of tasks completed correctly |
| **Steps to Completion** | Efficiency metric |
| **Tool Call Accuracy** | Correct tool selected + correct args |
| **Hallucination Rate** | % of outputs with factual errors |
| **Latency P50/P95** | Responsiveness |
| **Cost per Task** | Economic efficiency |

In [1]:
# ── LangSmith Setup ───────────────────────────────────────────────────────────
import os

# Set these environment variables:
# export LANGCHAIN_TRACING_V2=true
# export LANGCHAIN_API_KEY=your_key
# export LANGCHAIN_PROJECT=my-agent-project

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "demo-agent"
# os.environ["LANGCHAIN_API_KEY"] = "your_key"

# After this, ALL LangChain/LangGraph runs are automatically traced
print("LangSmith tracing configured")
print("View traces at: https://smith.langchain.com")

LangSmith tracing configured
View traces at: https://smith.langchain.com


In [2]:
# ── Manual Tracing with LangSmith SDK ────────────────────────────────────────
# pip install langsmith
from langsmith import traceable, Client

ls_client = Client()

@traceable(name="my_llm_call", run_type="llm")
def call_llm(prompt: str, model: str = "gpt-4o-mini") -> str:
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )
    return response.choices[0].message.content

@traceable(name="my_agent_run")
def agent_pipeline(user_input: str) -> str:
    # Step 1: Plan
    plan = call_llm(f"Create a brief plan to answer: {user_input}")
    # Step 2: Execute
    answer = call_llm(f"Based on this plan: {plan}\nAnswer: {user_input}")
    return answer

# result = agent_pipeline("Explain RAG in one paragraph")
# print(result)
print("Traceable functions defined runs will appear in LangSmith")

Traceable functions defined runs will appear in LangSmith


/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/langsmith/client.py:652: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [3]:
# ── Custom Cost Tracker ───────────────────────────────────────────────────────
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class TokenUsage:
    input_tokens: int = 0
    output_tokens: int = 0
    model: str = "gpt-4o-mini"
    
    PRICING = {
        "gpt-4o":          {"in": 2.50,  "out": 10.00},
        "gpt-4o-mini":     {"in": 0.15,  "out": 0.60},
        "claude-sonnet-4-6": {"in": 3.00, "out": 15.00},
        "claude-haiku-4-5":  {"in": 0.25, "out": 1.25},
    }
    
    @property
    def cost_usd(self) -> float:
        prices = self.PRICING.get(self.model, {"in": 1.0, "out": 3.0})
        return (self.input_tokens / 1_000_000) * prices["in"] + \
               (self.output_tokens / 1_000_000) * prices["out"]

class AgentCostTracker:
    def __init__(self):
        self.runs: list[dict] = []
    
    def record(self, run_id: str, usage: TokenUsage, task: str = ""):
        self.runs.append({
            "run_id": run_id,
            "timestamp": datetime.now().isoformat(),
            "model": usage.model,
            "input_tokens": usage.input_tokens,
            "output_tokens": usage.output_tokens,
            "cost_usd": usage.cost_usd,
            "task": task
        })
    
    def summary(self) -> dict:
        total_cost = sum(r["cost_usd"] for r in self.runs)
        total_input = sum(r["input_tokens"] for r in self.runs)
        total_output = sum(r["output_tokens"] for r in self.runs)
        return {
            "total_runs": len(self.runs),
            "total_cost_usd": round(total_cost, 6),
            "total_input_tokens": total_input,
            "total_output_tokens": total_output,
            "avg_cost_per_run": round(total_cost / max(len(self.runs), 1), 6)
        }

# Demo
tracker = AgentCostTracker()
tracker.record("run-001", TokenUsage(500, 100, "gpt-4o-mini"), "answer question")
tracker.record("run-002", TokenUsage(2000, 500, "gpt-4o"), "complex analysis")
tracker.record("run-003", TokenUsage(800, 200, "claude-sonnet-4-6"), "code review")

print("Cost Summary:")
import json
print(json.dumps(tracker.summary(), indent=2))

Cost Summary:
{
  "total_runs": 3,
  "total_cost_usd": 0.015535,
  "total_input_tokens": 3300,
  "total_output_tokens": 800,
  "avg_cost_per_run": 0.005178
}


In [4]:
# ── Simple Agent Evaluation Framework ────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class EvalCase:
    input: str
    expected_keywords: list[str]
    should_use_tools: bool = False

def evaluate_agent(agent_fn, eval_cases: list[EvalCase]) -> dict:
    results = []
    
    for case in eval_cases:
        output = agent_fn(case.input)
        
        # Check keyword coverage
        keywords_found = sum(
            1 for kw in case.expected_keywords 
            if kw.lower() in output.lower()
        )
        keyword_score = keywords_found / len(case.expected_keywords)
        
        results.append({
            "input": case.input,
            "keyword_score": keyword_score,
            "passed": keyword_score >= 0.8
        })
    
    passed = sum(1 for r in results if r["passed"])
    return {
        "total": len(results),
        "passed": passed,
        "pass_rate": passed / len(results),
        "results": results
    }

# Simple agent to evaluate
def simple_agent(question: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        max_tokens=200
    )
    return r.choices[0].message.content

eval_cases = [
    EvalCase("What is gradient descent?", ["gradient", "learning rate", "optimization"]),
    EvalCase("What is a neural network?", ["neurons", "layers", "weights"]),
]

# results = evaluate_agent(simple_agent, eval_cases)
# print(json.dumps(results, indent=2))
print("Evaluation framework ready uncomment evaluate_agent call to run")

Evaluation framework ready uncomment evaluate_agent call to run


## Additional Learning Resources

### Tools
- [LangSmith Docs](https://docs.smith.langchain.com/)
- [AgentOps](https://www.agentops.ai/)
- [Arize Phoenix](https://phoenix.arize.com/)
- [OpenLLMetry (GitHub)](https://github.com/traceloop/openllmetry)
- [Helicone](https://www.helicone.ai/) LLM observability

### Standards
- [OpenTelemetry for LLMs (OTel Gen AI)](https://opentelemetry.io/docs/specs/semconv/gen-ai/)

### Evaluation Frameworks
- [RAGAS](https://github.com/explodinggradients/ragas)
- [DeepEval](https://github.com/confident-ai/deepeval)
- [promptfoo](https://github.com/promptfoo/promptfoo)